# Experiment 6: Two-Dimensional Plate Conduction

**Aim:** Compare Jacobi and line-TDMA/SOR solutions of the steady two-dimensional Laplace equation against an analytical solution.

**Numerical method:** Finite-difference Laplace discretization with point Jacobi and line-implicit TDMA/SOR iteration.

> **Scope:** The comparison focuses on accuracy, convergence rate, residual definitions, and grid dependence.

This Google Colab notebook is self-contained. It reconstructs the tested solver inside the temporary Colab runtime, runs it, displays quantitative plots and an inline field contour, and packages every output file for download.

> **Colab workflow:** Choose a run mode, use **Runtime > Run all**, then download the results ZIP from the final cell. `quick` is only a workflow check; use `lab` or an approved `custom` case for submission.

In [ ]:
#@title 1. Prepare the self-contained Colab runtime

from pathlib import Path
import shutil
import subprocess
import sys
import zipfile
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

IN_COLAB = False
try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    colab_files = None


def run_solver(script_path, arguments, output_dir, timeout=1800):
    output_dir = Path(output_dir)
    if output_dir.exists():
        shutil.rmtree(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    command = [sys.executable, str(script_path), *[str(value) for value in arguments]]
    print("Running:", " ".join(command))
    completed = subprocess.run(command, cwd=output_dir, capture_output=True, text=True, timeout=timeout)
    (output_dir / "solver_stdout.log").write_text(completed.stdout, encoding="utf-8")
    (output_dir / "solver_stderr.log").write_text(completed.stderr, encoding="utf-8")
    if completed.stdout.strip():
        print(completed.stdout.strip())
    if completed.returncode != 0:
        raise RuntimeError(completed.stderr.strip() or f"Solver exited with code {completed.returncode}")
    if completed.stderr.strip():
        print(completed.stderr.strip())
    return output_dir


def read_csv(path):
    data = np.genfromtxt(path, delimiter=",", names=True, dtype=None, encoding="utf-8")
    return np.atleast_1d(data)


def show_summary(path):
    text = Path(path).read_text(encoding="utf-8").strip()
    display(Markdown(f"### Solver summary\n```text\n{text}\n```"))
    return text


def list_outputs(output_dir):
    rows = []
    for path in sorted(Path(output_dir).iterdir()):
        if path.is_file():
            size = path.stat().st_size
            if size < 1024:
                size_text = f"{size} B"
            elif size < 1024**2:
                size_text = f"{size / 1024:.1f} KB"
            else:
                size_text = f"{size / 1024**2:.1f} MB"
            rows.append((path.name, size_text))
    table = "\n".join(f"| `{name}` | {size} |" for name, size in rows)
    display(Markdown(f"### Generated files\n| File | Size |\n|---|---:|\n{table}"))
    return rows


def read_structured_vtk(path):
    lines = Path(path).read_text(encoding="utf-8").splitlines()
    dims_index = next(i for i, line in enumerate(lines) if line.startswith("DIMENSIONS "))
    _, nx, ny, nz = lines[dims_index].split()
    nx, ny, nz = int(nx), int(ny), int(nz)
    if nz != 1:
        raise ValueError("Only 2D structured VTK files are supported by this notebook preview")
    points_index = next(i for i, line in enumerate(lines) if line.startswith("POINTS "))
    npoints = int(lines[points_index].split()[1])
    points = np.array([[float(v) for v in lines[points_index + 1 + k].split()[:3]] for k in range(npoints)])
    x = points[:, 0].reshape(ny, nx)
    y = points[:, 1].reshape(ny, nx)
    data_index = next(i for i, line in enumerate(lines) if line.startswith("POINT_DATA ")) + 1
    scalars = {}
    vectors = {}
    i = data_index
    while i < len(lines):
        if lines[i].startswith("SCALARS "):
            name = lines[i].split()[1]
            i += 2
            values = np.array([float(lines[i + k].split()[0]) for k in range(npoints)])
            scalars[name] = values.reshape(ny, nx)
            i += npoints
        elif lines[i].startswith("VECTORS "):
            name = lines[i].split()[1]
            i += 1
            values = np.array([[float(v) for v in lines[i + k].split()[:3]] for k in range(npoints)])
            vectors[name] = (values[:, 0].reshape(ny, nx), values[:, 1].reshape(ny, nx))
            i += npoints
        else:
            i += 1
    return x, y, scalars, vectors


def plot_vtk_scalar(path, preferred_field=None):
    path = Path(path)
    if not path.is_file():
        print(f"VTK preview skipped: {path.name} was not generated")
        return
    x, y, scalars, _ = read_structured_vtk(path)
    if not scalars:
        print("VTK preview skipped: no scalar field found")
        return
    field = preferred_field if preferred_field in scalars else next(iter(scalars))
    plt.figure(figsize=(8, 4.8))
    image = plt.contourf(x, y, scalars[field], levels=30)
    plt.colorbar(image, label=field.replace("_", " "))
    plt.xlabel("x")
    plt.ylabel("y")
    plt.title(f"{field.replace('_', ' ').title()} - inline field preview")
    plt.axis("equal")
    plt.tight_layout()
    plt.show()


def make_results_zip(output_dir, experiment_number):
    output_dir = Path(output_dir)
    archive = output_dir.parent / f"experiment_{experiment_number:02d}_results.zip"
    if archive.exists():
        archive.unlink()
    with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as handle:
        for path in sorted(output_dir.rglob("*")):
            if path.is_file():
                handle.write(path, path.relative_to(output_dir))
    print(f"Results archive: {archive} ({archive.stat().st_size / 1024:.1f} KB)")
    return archive


EXPERIMENT = 6
ROOT = Path("/content/cfd_lab_exp06") if IN_COLAB else Path.cwd() / "_colab_runtime_exp06"
CODE_DIR = ROOT / "code"
OUTPUT_DIR = ROOT / "outputs"
CODE_DIR.mkdir(parents=True, exist_ok=True)
(CODE_DIR / "common.py").write_text('from __future__ import annotations\n\nimport csv\nimport math\nfrom pathlib import Path\nfrom typing import Iterable, Mapping\n\nimport numpy as np\n\n\ndef trapezoidal_integral(values: np.ndarray, coordinates: np.ndarray) -> float:\n    y = np.asarray(values, dtype=float).ravel()\n    x = np.asarray(coordinates, dtype=float).ravel()\n    if y.size != x.size or y.size < 2:\n        raise ValueError("trapezoidal integration requires equally sized arrays with at least two points")\n    return float(np.sum(0.5 * (y[1:] + y[:-1]) * np.diff(x)))\n\n\ndef write_csv(path: str | Path, header: Iterable[str], rows: Iterable[Iterable[object]]) -> None:\n    with Path(path).open("w", newline="", encoding="utf-8") as handle:\n        writer = csv.writer(handle)\n        writer.writerow(list(header))\n        writer.writerows(rows)\n\n\ndef mesh_xy(nx: int, ny: int, lx: float = 1.0, ly: float = 1.0, x0: float = 0.0, y0: float = 0.0) -> tuple[np.ndarray, np.ndarray]:\n    x = np.linspace(x0, x0 + lx, nx)\n    y = np.linspace(y0, y0 + ly, ny)\n    return np.meshgrid(x, y)\n\n\ndef write_structured_grid_vtk(\n    path: str | Path,\n    x: np.ndarray,\n    y: np.ndarray,\n    scalars: Mapping[str, np.ndarray] | None = None,\n    vectors: Mapping[str, tuple[np.ndarray, np.ndarray]] | None = None,\n    title: str = "CFD lab output",\n) -> None:\n    x = np.asarray(x, dtype=float)\n    y = np.asarray(y, dtype=float)\n    if x.shape != y.shape or x.ndim != 2:\n        raise ValueError("x and y must be equally shaped 2D arrays")\n    ny, nx = x.shape\n    scalar_map = dict(scalars or {})\n    vector_map = dict(vectors or {})\n    for name, values in scalar_map.items():\n        if np.asarray(values).shape != x.shape:\n            raise ValueError(f"scalar {name} has shape {np.asarray(values).shape}, expected {x.shape}")\n    for name, values in vector_map.items():\n        if len(values) != 2 or np.asarray(values[0]).shape != x.shape or np.asarray(values[1]).shape != x.shape:\n            raise ValueError(f"vector {name} has incompatible shape")\n    with Path(path).open("w", encoding="utf-8") as handle:\n        handle.write("# vtk DataFile Version 3.0\\n")\n        handle.write(f"{title}\\n")\n        handle.write("ASCII\\n")\n        handle.write("DATASET STRUCTURED_GRID\\n")\n        handle.write(f"DIMENSIONS {nx} {ny} 1\\n")\n        handle.write(f"POINTS {nx * ny} float\\n")\n        for j in range(ny):\n            for i in range(nx):\n                handle.write(f"{x[j, i]:.12e} {y[j, i]:.12e} 0\\n")\n        handle.write(f"POINT_DATA {nx * ny}\\n")\n        for name, values in scalar_map.items():\n            data = np.asarray(values, dtype=float)\n            handle.write(f"SCALARS {name} float 1\\n")\n            handle.write("LOOKUP_TABLE default\\n")\n            for value in data.ravel(order="C"):\n                handle.write(f"{value:.12e}\\n")\n        for name, values in vector_map.items():\n            vx = np.asarray(values[0], dtype=float)\n            vy = np.asarray(values[1], dtype=float)\n            handle.write(f"VECTORS {name} float\\n")\n            for ax, ay in zip(vx.ravel(order="C"), vy.ravel(order="C")):\n                handle.write(f"{ax:.12e} {ay:.12e} 0\\n")\n\n\ndef write_polyline_vtk(path: str | Path, x: np.ndarray, y: np.ndarray, title: str = "polyline", close: bool = False) -> None:\n    x = np.asarray(x, dtype=float).ravel()\n    y = np.asarray(y, dtype=float).ravel()\n    if x.size != y.size or x.size < 2:\n        raise ValueError("polyline requires equally sized x and y arrays")\n    ids = list(range(x.size))\n    if close:\n        ids.append(0)\n    with Path(path).open("w", encoding="utf-8") as handle:\n        handle.write("# vtk DataFile Version 3.0\\n")\n        handle.write(f"{title}\\n")\n        handle.write("ASCII\\n")\n        handle.write("DATASET POLYDATA\\n")\n        handle.write(f"POINTS {x.size} float\\n")\n        for xi, yi in zip(x, y):\n            handle.write(f"{xi:.12e} {yi:.12e} 0\\n")\n        handle.write(f"LINES 1 {len(ids) + 1}\\n")\n        handle.write(f"{len(ids)} {\' \'.join(str(i) for i in ids)}\\n")\n\n\ndef write_multiline_vtk(path: str | Path, lines: list[tuple[np.ndarray, np.ndarray]], title: str = "lines") -> None:\n    points: list[tuple[float, float]] = []\n    connectivity: list[list[int]] = []\n    for x, y in lines:\n        xa = np.asarray(x, dtype=float).ravel()\n        ya = np.asarray(y, dtype=float).ravel()\n        if xa.size != ya.size or xa.size < 2:\n            raise ValueError("each line needs equally sized x and y arrays")\n        ids = list(range(len(points), len(points) + xa.size))\n        points.extend(zip(xa.tolist(), ya.tolist()))\n        connectivity.append(ids)\n    size = sum(len(ids) + 1 for ids in connectivity)\n    with Path(path).open("w", encoding="utf-8") as handle:\n        handle.write("# vtk DataFile Version 3.0\\n")\n        handle.write(f"{title}\\n")\n        handle.write("ASCII\\n")\n        handle.write("DATASET POLYDATA\\n")\n        handle.write(f"POINTS {len(points)} float\\n")\n        for xi, yi in points:\n            handle.write(f"{xi:.12e} {yi:.12e} 0\\n")\n        handle.write(f"LINES {len(connectivity)} {size}\\n")\n        for ids in connectivity:\n            handle.write(f"{len(ids)} {\' \'.join(str(i) for i in ids)}\\n")\n\n\ndef write_points_vtk(path: str | Path, x: np.ndarray, y: np.ndarray, scalars: Mapping[str, np.ndarray] | None = None, title: str = "points") -> None:\n    xa = np.asarray(x, dtype=float).ravel()\n    ya = np.asarray(y, dtype=float).ravel()\n    if xa.size != ya.size:\n        raise ValueError("point arrays must have equal size")\n    scalar_map = dict(scalars or {})\n    with Path(path).open("w", encoding="utf-8") as handle:\n        handle.write("# vtk DataFile Version 3.0\\n")\n        handle.write(f"{title}\\n")\n        handle.write("ASCII\\n")\n        handle.write("DATASET POLYDATA\\n")\n        handle.write(f"POINTS {xa.size} float\\n")\n        for xi, yi in zip(xa, ya):\n            handle.write(f"{xi:.12e} {yi:.12e} 0\\n")\n        handle.write(f"VERTICES {xa.size} {2 * xa.size}\\n")\n        for i in range(xa.size):\n            handle.write(f"1 {i}\\n")\n        if scalar_map:\n            handle.write(f"POINT_DATA {xa.size}\\n")\n            for name, values in scalar_map.items():\n                data = np.asarray(values, dtype=float).ravel()\n                if data.size != xa.size:\n                    raise ValueError(f"scalar {name} has wrong size")\n                handle.write(f"SCALARS {name} float 1\\n")\n                handle.write("LOOKUP_TABLE default\\n")\n                for value in data:\n                    handle.write(f"{value:.12e}\\n")\n\n\ndef gradient(field: np.ndarray, dx: float, dy: float) -> tuple[np.ndarray, np.ndarray]:\n    gy, gx = np.gradient(field, dy, dx, edge_order=2)\n    return gx, gy\n\n\ndef laplacian(field: np.ndarray, dx: float, dy: float) -> np.ndarray:\n    out = np.zeros_like(field)\n    out[1:-1, 1:-1] = (\n        (field[1:-1, 2:] - 2.0 * field[1:-1, 1:-1] + field[1:-1, :-2]) / (dx * dx)\n        + (field[2:, 1:-1] - 2.0 * field[1:-1, 1:-1] + field[:-2, 1:-1]) / (dy * dy)\n    )\n    return out\n\n\ndef upwind_derivative_x(field: np.ndarray, velocity: np.ndarray, dx: float) -> np.ndarray:\n    out = np.zeros_like(field)\n    backward = (field[1:-1, 1:-1] - field[1:-1, :-2]) / dx\n    forward = (field[1:-1, 2:] - field[1:-1, 1:-1]) / dx\n    out[1:-1, 1:-1] = np.where(velocity[1:-1, 1:-1] >= 0.0, backward, forward)\n    return out\n\n\ndef upwind_derivative_y(field: np.ndarray, velocity: np.ndarray, dy: float) -> np.ndarray:\n    out = np.zeros_like(field)\n    backward = (field[1:-1, 1:-1] - field[:-2, 1:-1]) / dy\n    forward = (field[2:, 1:-1] - field[1:-1, 1:-1]) / dy\n    out[1:-1, 1:-1] = np.where(velocity[1:-1, 1:-1] >= 0.0, backward, forward)\n    return out\n\n\ndef poisson_jacobi(phi: np.ndarray, rhs: np.ndarray, dx: float, dy: float, iterations: int, boundary: str = "neumann", mask: np.ndarray | None = None) -> np.ndarray:\n    dx2 = dx * dx\n    dy2 = dy * dy\n    denom = 2.0 * (dx2 + dy2)\n    for _ in range(iterations):\n        old = phi.copy()\n        phi[1:-1, 1:-1] = (\n            (old[1:-1, 2:] + old[1:-1, :-2]) * dy2\n            + (old[2:, 1:-1] + old[:-2, 1:-1]) * dx2\n            - rhs[1:-1, 1:-1] * dx2 * dy2\n        ) / denom\n        if mask is not None:\n            phi[mask > 0.5] = 0.0\n        if boundary == "neumann":\n            phi[:, 0] = phi[:, 1]\n            phi[:, -1] = 0.0\n            phi[0, :] = phi[1, :]\n            phi[-1, :] = phi[-2, :]\n        elif boundary == "closed":\n            phi[:, 0] = phi[:, 1]\n            phi[:, -1] = phi[:, -2]\n            phi[0, :] = phi[1, :]\n            phi[-1, :] = phi[-2, :]\n            phi -= np.mean(phi)\n        else:\n            raise ValueError("unsupported pressure boundary")\n    return phi\n\n\ndef tdma(a: np.ndarray, b: np.ndarray, c: np.ndarray, d: np.ndarray) -> np.ndarray:\n    a = np.asarray(a, dtype=float).copy()\n    b = np.asarray(b, dtype=float).copy()\n    c = np.asarray(c, dtype=float).copy()\n    d = np.asarray(d, dtype=float).copy()\n    n = d.size\n    if n == 0 or a.size != n or b.size != n or c.size != n:\n        raise ValueError("TDMA arrays must have equal nonzero size")\n    for i in range(1, n):\n        if abs(b[i - 1]) < 1e-30:\n            raise FloatingPointError("zero TDMA pivot")\n        factor = a[i] / b[i - 1]\n        b[i] -= factor * c[i - 1]\n        d[i] -= factor * d[i - 1]\n    x = np.empty(n, dtype=float)\n    if abs(b[-1]) < 1e-30:\n        raise FloatingPointError("zero TDMA pivot")\n    x[-1] = d[-1] / b[-1]\n    for i in range(n - 2, -1, -1):\n        if abs(b[i]) < 1e-30:\n            raise FloatingPointError("zero TDMA pivot")\n        x[i] = (d[i] - c[i] * x[i + 1]) / b[i]\n    return x\n\n\ndef relative_l2(numerical: np.ndarray, reference: np.ndarray) -> float:\n    a = np.asarray(numerical, dtype=float)\n    b = np.asarray(reference, dtype=float)\n    denominator = float(np.linalg.norm(b.ravel()))\n    numerator = float(np.linalg.norm((a - b).ravel()))\n    return numerator / max(denominator, 1e-30)\n\n\ndef require_finite(*arrays: np.ndarray) -> None:\n    for array in arrays:\n        if not np.all(np.isfinite(array)):\n            raise FloatingPointError("non-finite numerical field detected")\n\n\ndef positive_float(value: str, name: str) -> float:\n    result = float(value)\n    if not math.isfinite(result) or result <= 0.0:\n        raise ValueError(f"{name} must be positive and finite")\n    return result\n\n\ndef positive_int(value: str, name: str, minimum: int = 1) -> int:\n    result = int(value)\n    if result < minimum:\n        raise ValueError(f"{name} must be at least {minimum}")\n    return result\n', encoding="utf-8")
(CODE_DIR / 'exp06_plate_conduction_explicit_implicit.py').write_text('from __future__ import annotations\n\nimport csv\nimport math\nimport sys\n\nimport numpy as np\n\nfrom common import relative_l2, tdma, write_multiline_vtk, write_structured_grid_vtk\n\n\ndef analytical_solution(x: np.ndarray, y: np.ndarray, length: float, height: float, t_left: float) -> np.ndarray:\n    denominator = math.sinh(math.pi * length / height)\n    return t_left * np.sinh(math.pi * (length - x) / height) * np.sin(math.pi * y / height) / denominator\n\n\ndef apply_boundaries(field: np.ndarray, y: np.ndarray, height: float, t_left: float) -> None:\n    field[:, 0] = t_left * np.sin(math.pi * y / height)\n    field[:, -1] = 0.0\n    field[0, :] = 0.0\n    field[-1, :] = 0.0\n\n\ndef main() -> int:\n    nx = int(sys.argv[1]) if len(sys.argv) > 1 else 81\n    ny = int(sys.argv[2]) if len(sys.argv) > 2 else 61\n    length = float(sys.argv[3]) if len(sys.argv) > 3 else 1.0\n    height = float(sys.argv[4]) if len(sys.argv) > 4 else 1.0\n    t_left = float(sys.argv[5]) if len(sys.argv) > 5 else 100.0\n    max_iterations = int(sys.argv[6]) if len(sys.argv) > 6 else 10000\n    tolerance = float(sys.argv[7]) if len(sys.argv) > 7 else 1e-8\n    omega = float(sys.argv[8]) if len(sys.argv) > 8 else 1.55\n    screen_every = int(sys.argv[9]) if len(sys.argv) > 9 else 1000\n    if nx < 5 or ny < 5 or length <= 0.0 or height <= 0.0 or max_iterations < 1 or tolerance <= 0.0 or not 0.0 < omega < 2.0:\n        raise ValueError("usage: python exp06_plate_conduction_explicit_implicit.py nx ny L H Tleft max_iterations tolerance omega screen_every")\n    x1 = np.linspace(0.0, length, nx)\n    y1 = np.linspace(0.0, height, ny)\n    x, y = np.meshgrid(x1, y1)\n    dx = x1[1] - x1[0]\n    dy = y1[1] - y1[0]\n    beta = (dx / dy) ** 2\n    analytical = analytical_solution(x, y, length, height, t_left)\n    jacobi = np.zeros((ny, nx), dtype=float)\n    line_tdma = np.zeros((ny, nx), dtype=float)\n    apply_boundaries(jacobi, y1, height, t_left)\n    apply_boundaries(line_tdma, y1, height, t_left)\n    history: list[list[object]] = []\n    jacobi_iterations = max_iterations\n    jacobi_residual = float("inf")\n    for iteration in range(1, max_iterations + 1):\n        old = jacobi.copy()\n        jacobi[1:-1, 1:-1] = (old[1:-1, 2:] + old[1:-1, :-2] + beta * (old[2:, 1:-1] + old[:-2, 1:-1])) / (2.0 * (1.0 + beta))\n        apply_boundaries(jacobi, y1, height, t_left)\n        jacobi_residual = float(np.max(np.abs(jacobi - old)))\n        history.append(["jacobi", iteration, jacobi_residual, relative_l2(jacobi, analytical)])\n        if iteration == 1 or iteration == max_iterations or iteration % max(1, screen_every) == 0:\n            print(f"jacobi_iteration={iteration} residual={jacobi_residual:.6e}")\n        if jacobi_residual < tolerance:\n            jacobi_iterations = iteration\n            break\n    tdma_iterations = max_iterations\n    tdma_residual = float("inf")\n    n_unknown = nx - 2\n    for iteration in range(1, max_iterations + 1):\n        old_iteration = line_tdma.copy()\n        for j in range(1, ny - 1):\n            a = np.full(n_unknown, -1.0)\n            b = np.full(n_unknown, 2.0 * (1.0 + beta))\n            c = np.full(n_unknown, -1.0)\n            rhs = beta * (line_tdma[j + 1, 1:-1] + line_tdma[j - 1, 1:-1])\n            rhs[0] += line_tdma[j, 0]\n            rhs[-1] += line_tdma[j, -1]\n            a[0] = 0.0\n            c[-1] = 0.0\n            solved = tdma(a, b, c, rhs)\n            line_tdma[j, 1:-1] = (1.0 - omega) * line_tdma[j, 1:-1] + omega * solved\n        apply_boundaries(line_tdma, y1, height, t_left)\n        tdma_residual = float(np.max(np.abs(line_tdma - old_iteration)))\n        history.append(["line_tdma", iteration, tdma_residual, relative_l2(line_tdma, analytical)])\n        if iteration == 1 or iteration == max_iterations or iteration % max(1, screen_every) == 0:\n            print(f"line_tdma_iteration={iteration} residual={tdma_residual:.6e}")\n        if tdma_residual < tolerance:\n            tdma_iterations = iteration\n            break\n    jacobi_error = relative_l2(jacobi, analytical)\n    tdma_error = relative_l2(line_tdma, analytical)\n    write_structured_grid_vtk("plate_conduction_2d.vtk", x, y, {"temperature_jacobi": jacobi, "temperature_line_tdma": line_tdma, "temperature_analytical": analytical, "jacobi_absolute_error": np.abs(jacobi - analytical), "line_tdma_absolute_error": np.abs(line_tdma - analytical)}, title="2D plate heat conduction")\n    write_multiline_vtk("plate_boundaries.vtk", [(np.array([0.0, length]), np.array([0.0, 0.0])), (np.array([length, length]), np.array([0.0, height])), (np.array([length, 0.0]), np.array([height, height])), (np.array([0.0, 0.0]), np.array([height, 0.0]))], "Plate boundaries")\n    center_j = ny // 2\n    center_i = nx // 2\n    with open("plate_conduction_profiles.csv", "w", newline="", encoding="utf-8") as handle:\n        writer = csv.writer(handle)\n        writer.writerow(["coordinate", "horizontal_jacobi", "horizontal_line_tdma", "horizontal_analytical", "vertical_jacobi", "vertical_line_tdma", "vertical_analytical"])\n        nrows = max(nx, ny)\n        for k in range(nrows):\n            coordinate = k / max(nrows - 1, 1)\n            ih = min(int(round(coordinate * (nx - 1))), nx - 1)\n            jv = min(int(round(coordinate * (ny - 1))), ny - 1)\n            writer.writerow([coordinate, jacobi[center_j, ih], line_tdma[center_j, ih], analytical[center_j, ih], jacobi[jv, center_i], line_tdma[jv, center_i], analytical[jv, center_i]])\n    with open("plate_conduction_history.csv", "w", newline="", encoding="utf-8") as handle:\n        writer = csv.writer(handle)\n        writer.writerow(["method", "iteration", "residual_linf", "relative_l2_error"])\n        writer.writerows(history)\n    with open("plate_conduction_summary.txt", "w", encoding="utf-8") as handle:\n        handle.write("Steady 2D Laplace heat conduction\\n")\n        handle.write(f"nx {nx}\\nny {ny}\\nL {length:.12e}\\nH {height:.12e}\\nTleft {t_left:.12e}\\n")\n        handle.write(f"jacobi_iterations {jacobi_iterations}\\njacobi_final_residual {jacobi_residual:.12e}\\njacobi_relative_L2_error {jacobi_error:.12e}\\n")\n        handle.write(f"line_tdma_iterations {tdma_iterations}\\nline_tdma_final_residual {tdma_residual:.12e}\\nline_tdma_relative_L2_error {tdma_error:.12e}\\n")\n    print(f"Jacobi iterations={jacobi_iterations} residual={jacobi_residual:.6e} L2_error={jacobi_error:.6e}")\n    print(f"Line_TDMA iterations={tdma_iterations} residual={tdma_residual:.6e} L2_error={tdma_error:.6e}")\n    print("Wrote plate_conduction_2d.vtk, plate_boundaries.vtk, plate_conduction_profiles.csv, plate_conduction_history.csv, plate_conduction_summary.txt")\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', encoding="utf-8")
SCRIPT = CODE_DIR / 'exp06_plate_conduction_explicit_implicit.py'
print("Runtime:", "Google Colab" if IN_COLAB else "local notebook test")
print("Working folder:", ROOT)
print("No package installation or external file upload is required.")


## Parameters

Use `quick` to test the workflow, `lab` for the prescribed case, and `custom` only when changing the form values for the assignment.

In [ ]:
#@title 2. Select run mode and parameters
RUN_MODE = "quick" #@param ["quick", "lab", "custom"]

nx = 81 #@param {type:"integer"}
ny = 61 #@param {type:"integer"}
L = 1.0 #@param {type:"number"}
H = 1.0 #@param {type:"number"}
T_left = 100.0 #@param {type:"number"}
max_iterations = 10000 #@param {type:"integer"}
tolerance = 1e-08 #@param {type:"number"}
omega = 1.55 #@param {type:"number"}
screen_every = 1000 #@param {type:"integer"}

QUICK_ARGS = [21, 21, 1.0, 1.0, 100, 30, 1e-05, 1.3, 10]
LAB_ARGS = [81, 61, 1.0, 1.0, 100, 10000, 1e-08, 1.55, 1000]
CUSTOM_ARGS = [nx, ny, L, H, T_left, max_iterations, tolerance, omega, screen_every]
ARGS = QUICK_ARGS if RUN_MODE == "quick" else LAB_ARGS if RUN_MODE == "lab" else CUSTOM_ARGS
print("Run mode:", RUN_MODE)
print("Arguments:", ARGS)


## Run the solver

In [ ]:
#@title 3. Run the numerical solver
OUT = run_solver(SCRIPT, ARGS, OUTPUT_DIR)


## Numerical summary

In [ ]:
#@title 4. Read the solver summary
summary_text = show_summary(OUT / 'plate_conduction_summary.txt')


## Quantitative plots

In [ ]:
#@title 5. Plot the primary numerical results
profiles = read_csv(OUT / "plate_conduction_profiles.csv")
plt.figure(figsize=(7, 4))
plt.plot(profiles["coordinate"], profiles["horizontal_jacobi"], label="Jacobi")
plt.plot(profiles["coordinate"], profiles["horizontal_line_tdma"], label="Line TDMA")
plt.plot(profiles["coordinate"], profiles["horizontal_analytical"], "--", label="Analytical")
plt.xlabel("Normalized coordinate")
plt.ylabel("Temperature")
plt.title("Horizontal centreline")
plt.legend()
plt.grid(True)
plt.show()

history = read_csv(OUT / "plate_conduction_history.csv")
plt.figure(figsize=(7, 4))
for method in np.unique(history["method"]):
    mask = history["method"] == method
    plt.semilogy(history["iteration"][mask], history["residual_linf"][mask], label=str(method))
plt.xlabel("Iteration")
plt.ylabel("L-infinity residual")
plt.title("Convergence comparison")
plt.legend()
plt.grid(True)
plt.show()

## Inline field visualization

This preview removes the need for ParaView during the basic laboratory workflow. The VTK file is still included in the downloaded results for optional advanced post-processing.

In [ ]:
#@title 6. Display the final field and generated files
plot_vtk_scalar(OUT / 'plate_conduction_2d.vtk', 'temperature_line_tdma')
files = list_outputs(OUT)


## Download results

In [ ]:
#@title 7. Package and download the results
DOWNLOAD_RESULTS = False #@param {type:"boolean"}
RESULTS_ZIP = make_results_zip(OUT, 6)
if DOWNLOAD_RESULTS:
    if IN_COLAB:
        colab_files.download(str(RESULTS_ZIP))
    else:
        print("Local test mode. Archive retained at:", RESULTS_ZIP)
else:
    print("Set DOWNLOAD_RESULTS to True and run this cell when ready to download.")


## 7. Compulsory assignment

Compare Jacobi and line-TDMA/SOR for at least two grids. Report iterations, residual, analytical error, and the effect of relaxation factor.

## Report checklist

- State the governing equations and boundary conditions.
- Report the complete parameter set and grid.
- Include residual or error history.
- Include the required quantitative comparison.
- Include at least one correctly labelled inline field figure; VTK/ParaView figures are optional unless specifically assigned.
- Distinguish observed numerical behaviour from physical interpretation.
- State the limitations of the supplied teaching model.